# Démonstration Transport : Blob de Biomasse avec Île

Ce notebook démontre le transport physique (advection + diffusion) d'un blob de biomasse
qui contourne une île au milieu du domaine.

**Méthodes utilisées** :

-   Advection : Flux-based upwind (volumes finis)
-   Diffusion : Euler explicite sur grille plane
-   Masquage : Île = flux nul aux interfaces terre/mer


In [ ]:
# Imports
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from seapopym_message.transport import (
    PlaneGrid,
    BoundaryConditions,
    BoundaryType,
    advection_upwind_flux,
    diffusion_explicit_spherical,
)

print("Modules importés avec succès !")

## 1. Configuration du domaine et de la grille


In [ ]:
# Paramètres de la grille
nlat, nlon = 100, 100  # Grille 100x100
dx = 10e3  # 10 km spacing
dy = 10e3  # 10 km spacing

# Créer la grille plane
grid = PlaneGrid(dx=dx, dy=dy, nlat=nlat, nlon=nlon)

# Conditions aux limites : fermées (flux nul aux bords)
boundary = BoundaryConditions(
    north=BoundaryType.CLOSED,
    south=BoundaryType.CLOSED,
    east=BoundaryType.CLOSED,
    west=BoundaryType.CLOSED,
)

print(f"Grille: {nlat}x{nlon}, dx={dx / 1e3:.1f} km, dy={dy / 1e3:.1f} km")
print(f"Domaine: {nlat * dy / 1e3:.0f} km x {nlon * dx / 1e3:.0f} km")

## 2. Création du blob de biomasse initial et de l'île


In [ ]:
# Créer le blob de biomasse initial (gaussien)
x = np.arange(nlon)
y = np.arange(nlat)
X, Y = np.meshgrid(x, y)

# Position initiale du blob (à gauche)
blob_x0, blob_y0 = 20, 50  # Position (i, j)
blob_sigma = 10  # Largeur du blob

biomass_init = 100.0 * np.exp(-((X - blob_x0) ** 2 + (Y - blob_y0) ** 2) / (2 * blob_sigma**2))

# Créer l'île (masque) au milieu du domaine
island_x, island_y = 50, 50  # Centre de l'île
island_radius = 15  # Rayon de l'île

mask = np.ones((nlat, nlon), dtype=np.float32)  # 1 = océan
distance_to_island = np.sqrt((X - island_x) ** 2 + (Y - island_y) ** 2)
mask[distance_to_island <= island_radius] = 0.0  # 0 = terre

# Convertir en JAX arrays
biomass = jnp.array(biomass_init, dtype=jnp.float32)
mask_jax = jnp.array(mask, dtype=jnp.float32)

# Visualiser la configuration initiale
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Biomasse initiale
im1 = axes[0].imshow(biomass, origin="lower", cmap="YlOrRd", interpolation="bilinear")
axes[0].set_title("Biomasse Initiale [kg/m²]")
axes[0].set_xlabel("Longitude (index)")
axes[0].set_ylabel("Latitude (index)")
plt.colorbar(im1, ax=axes[0])

# Masque (île)
im2 = axes[1].imshow(mask, origin="lower", cmap="gray_r", interpolation="nearest")
axes[1].set_title("Masque (blanc=océan, noir=île)")
axes[1].set_xlabel("Longitude (index)")
axes[1].set_ylabel("Latitude (index)")
plt.colorbar(im2, ax=axes[1])

# Configuration combinée
im3 = axes[2].imshow(biomass * mask, origin="lower", cmap="YlOrRd", interpolation="bilinear")
axes[2].contour(mask, levels=[0.5], colors="black", linewidths=2)
axes[2].set_title("Biomasse + Île")
axes[2].set_xlabel("Longitude (index)")
axes[2].set_ylabel("Latitude (index)")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

print(f"Masse initiale : {np.sum(biomass * mask * dx * dy):.2e} kg")

## 3. Configuration du champ de vitesse (advection)

On crée un courant qui pousse la biomasse vers l'est (droite).


In [ ]:
# Champ de vitesse : courant vers l'est
u = jnp.ones((nlat, nlon), dtype=jnp.float32) * 0.2  # 0.2 m/s vers l'est
v = jnp.zeros((nlat, nlon), dtype=jnp.float32)  # Pas de déplacement N-S

# Coefficient de diffusion horizontale
D = 500.0  # m²/s (diffusion modérée)

# Pas de temps (doit respecter CFL pour stabilité)
dt = 300.0  # 300 s = 5 minutes

print(f"Vitesse : u={float(u[0, 0]):.2f} m/s (est), v={float(v[0, 0]):.2f} m/s")
print(f"Diffusion : D={D:.1f} m²/s")
print(f"Pas de temps : dt={dt:.0f} s")

# Vérifier la stabilité
cfl_adv = u[0, 0] * dt / dx
cfl_diff = D * dt / dx**2
print(f"\nCFL advection : {cfl_adv:.3f} (doit être < 1)")
print(f"CFL diffusion : {cfl_diff:.3f} (doit être < 0.25)")

if cfl_adv < 1 and cfl_diff < 0.25:
    print("✅ Paramètres stables !")
else:
    print("⚠️ Attention : paramètres potentiellement instables !")

## 4. Simulation du transport sur plusieurs pas de temps


In [ ]:
# Nombre de pas de temps
n_steps = 6000  # 200 steps * 5 min = 16.7 heures

# Stocker l'évolution
biomass_history = [np.array(biomass)]
mass_history = [float(jnp.sum(biomass * mask_jax * dx * dy))]
time_history = [0.0]

# Boucle de transport
current_biomass = biomass

print(f"Simulation de {n_steps} pas de temps...")
print(f"Durée totale : {n_steps * dt / 3600:.1f} heures\n")

for step in range(n_steps):
    # Advection
    biomass_advected = advection_upwind_flux(
        biomass=current_biomass,
        u=u,
        v=v,
        dt=dt,
        grid=grid,
        boundary=boundary,
        mask=mask_jax,
    )

    # Diffusion
    biomass_new = diffusion_explicit_spherical(
        biomass=biomass_advected,
        D=D,
        dt=dt,
        grid=grid,
        boundary=boundary,
        mask=mask_jax,
    )

    current_biomass = biomass_new

    # Sauvegarder tous les 10 pas
    if (step + 1) % 10 == 0:
        biomass_history.append(np.array(current_biomass))
        mass = float(jnp.sum(current_biomass * mask_jax * dx * dy))
        mass_history.append(mass)
        time_history.append((step + 1) * dt / 3600)  # Heures

        if (step + 1) % 50 == 0:
            conservation = mass / mass_history[0] * 100
            print(
                f"Step {step + 1}/{n_steps} - t={time_history[-1]:.1f}h - "
                f"Conservation: {conservation:.4f}%"
            )

print("\n✅ Simulation terminée !")

## 5. Analyse de la conservation de la masse


In [ ]:
# Calculer la conservation
mass_array = np.array(mass_history)
conservation = mass_array / mass_array[0] * 100

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(time_history, mass_array / 1e9, "b-", linewidth=2)
plt.axhline(mass_array[0] / 1e9, color="r", linestyle="--", label="Masse initiale")
plt.xlabel("Temps [heures]")
plt.ylabel("Masse totale [10⁹ kg]")
plt.title("Évolution de la masse totale")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(time_history, conservation, "g-", linewidth=2)
plt.axhline(100, color="r", linestyle="--", label="Conservation parfaite")
plt.axhline(99, color="orange", linestyle=":", label="Seuil 99%")
plt.xlabel("Temps [heures]")
plt.ylabel("Conservation [%]")
plt.title("Conservation de la masse")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print(f"Conservation finale : {conservation[-1]:.6f}%")
print(f"Perte/Gain de masse : {(conservation[-1] - 100):.6f}%")

## 6. Visualisation de l'évolution du blob

On visualise le blob à différents instants pour voir comment il contourne l'île.


In [ ]:
# Sélectionner quelques snapshots
n_snapshots = 6
snapshot_indices = np.linspace(0, len(biomass_history) - 1, n_snapshots, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Trouver le max pour une échelle cohérente
vmax = max(b.max() for b in biomass_history) / 10

for i, idx in enumerate(snapshot_indices):
    biomass_snap = biomass_history[idx]
    time_snap = time_history[idx]

    # Afficher la biomasse
    im = axes[i].imshow(
        biomass_snap * mask,
        origin="lower",
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        interpolation="bilinear",
    )

    # Contour de l'île
    axes[i].contour(mask, levels=[0.5], colors="black", linewidths=2)

    axes[i].set_title(f"t = {time_snap:.1f} heures")
    axes[i].set_xlabel("Longitude (index)")
    axes[i].set_ylabel("Latitude (index)")
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.suptitle("Évolution du blob de biomasse contournant l'île", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 8. Analyse du blocage de flux par l'île

Vérifier que la biomasse ne traverse pas l'île.


In [ ]:
# Vérifier que l'île reste sans biomasse
final_biomass = biomass_history[-1]
island_biomass = final_biomass[mask == 0]  # Biomasse sur l'île

print(
    f"Biomasse sur l'île (finale) : min={island_biomass.min():.2e}, "
    f"max={island_biomass.max():.2e}, mean={island_biomass.mean():.2e}"
)

if np.all(island_biomass < 1e-6):
    print("✅ Le flux est correctement bloqué par l'île !")
else:
    print("⚠️ Attention : de la biomasse a pénétré l'île")

# Visualiser la différence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biomasse finale
im1 = axes[0].imshow(final_biomass, origin="lower", cmap="YlOrRd", interpolation="bilinear")
axes[0].contour(mask, levels=[0.5], colors="black", linewidths=2)
axes[0].set_title("Biomasse Finale")
axes[0].set_xlabel("Longitude (index)")
axes[0].set_ylabel("Latitude (index)")
plt.colorbar(im1, ax=axes[0])

# Zoom sur l'île
i_min, i_max = island_y - 25, island_y + 25
j_min, j_max = island_x - 25, island_x + 25
im2 = axes[1].imshow(
    final_biomass[i_min:i_max, j_min:j_max], origin="lower", cmap="YlOrRd", interpolation="bilinear"
)
axes[1].contour(mask[i_min:i_max, j_min:j_max], levels=[0.5], colors="black", linewidths=2)
axes[1].set_title("Zoom sur l'île")
axes[1].set_xlabel("Longitude (index relatif)")
axes[1].set_ylabel("Latitude (index relatif)")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()